# 01c — Real-World High Dimensions

## The One-Sentence Version
High-dimensional data isn't a theoretical curiosity — it's what you're working with every time you touch genomics, imaging, text, patient records, or simulation.

## What You'll Build Intuition For
- Where high-dimensional data shows up in practice
- What "dimension" means in each domain
- Why different domains face the curse differently

## Prerequisites
- [01a — Building Intuition](01a_building_intuition.ipynb) — what a dimension is
- [01b — The Curse of Dimensionality](01b_curse_of_dimensionality.ipynb) — why high dimensions are hostile

## The Story

In 01a we built comfort with the idea of dimensions. In 01b we saw why high dimensions are treacherous. But both notebooks used synthetic data — toy examples designed to make a point.

This notebook grounds everything in reality. We'll look at four domains where high-dimensional data isn't optional — it's the starting point. Images, text, patient records, and simulation parameters. In every case, the data *appears* to need hundreds or thousands of dimensions, but actually lives on a much smaller structure.

That gap — between the number of columns in your spreadsheet and the number of independent things actually going on — is the entire reason dimensionality reduction works. By the end of this notebook, you'll see the same pattern repeating across wildly different fields.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data, make_patient_data

apply_style()
rng = np.random.default_rng(42)

## Images Are Points in Pixel-Space

Every image is a grid of numbers. An 8×8 greyscale digit is 64 numbers — one per pixel. That makes each image a single point in 64-dimensional space.

The remarkable thing: images of the *same* digit cluster together in that 64D space, even though no one told the system what a "3" looks like.

In [ ]:
# Load digits
data, target, images = make_digit_data()

# Show 5 different "3"s — 5 points in 64D that are near each other
threes_idx = np.where(target == 3)[0][:5]

fig, axes = plt.subplots(1, 5, figsize=(12, 2.5))
for ax, idx in zip(axes, threes_idx):
    ax.imshow(images[idx], cmap="gray_r")
    ax.set_title(f"Point #{idx}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Five different "3"s — five different points in 64D space', fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Each image is a point in {data.shape[1]}D space")
print(f"We have {data.shape[0]} images total, across 10 digit classes")

In [ ]:
# PCA to 2D — the clusters jump out
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(data)

fig, ax = plt.subplots(figsize=(10, 8))
for digit in range(10):
    mask = target == digit
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=10, alpha=0.6,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("64D → 2D: digits cluster by identity")
ax.legend(title="Digit", markerscale=3, fontsize=9)
plt.tight_layout()
plt.savefig("visuals/01c_digits_pca.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Just 2 components capture {sum(pca_2d.explained_variance_ratio_):.1%} of variance")
print("Even this crude projection shows digit clusters forming.")

Even squashed down from 64 dimensions to just 2, the digit classes separate into visible clusters. The 0s, 1s, and 7s are especially distinct. This is the structure that dimensionality reduction exploits — the data *looks* 64-dimensional but behaves as if it's much lower.

## Text Is Points in Word-Space

Text data faces the curse in an especially dramatic way. In a "bag-of-words" representation, every unique word in the vocabulary becomes a dimension. A vocabulary of 50,000 words means each document is a point in 50,000-dimensional space.

And most of that space is empty — any given document uses only a tiny fraction of the full vocabulary.

In [ ]:
# A tiny bag-of-words example (no heavy NLP dependencies)
documents = [
    "the patient has high blood pressure and chest pain",
    "blood test results show elevated glucose levels",
    "the drone flew over the mountain at high altitude",
    "simulation results show high casualty evacuation times",
    "the patient blood pressure is normal after treatment",
]

# Build vocabulary
vocab = sorted(set(word for doc in documents for word in doc.split()))
word_to_idx = {w: i for i, w in enumerate(vocab)}

# Build document-term matrix
dtm = np.zeros((len(documents), len(vocab)))
for i, doc in enumerate(documents):
    for word in doc.split():
        dtm[i, word_to_idx[word]] += 1

print(f"5 documents × {len(vocab)} unique words = {dtm.shape} matrix")
print(f"Non-zero entries: {np.count_nonzero(dtm)} out of {dtm.size} ({np.count_nonzero(dtm)/dtm.size:.0%})")
print(f"\nMost entries are 0 — the matrix is extremely sparse.")

# Visualise the sparsity
fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(dtm, cmap="Blues", aspect="auto")
ax.set_xlabel("Word index (dimension)")
ax.set_ylabel("Document")
ax.set_yticks(range(5))
ax.set_yticklabels([f"Doc {i+1}" for i in range(5)])
ax.set_title(f"Bag-of-Words Matrix: each document is a point in {len(vocab)}D space")
plt.colorbar(im, ax=ax, label="Word count")
plt.tight_layout()
plt.savefig("visuals/01c_bag_of_words.png", dpi=150, bbox_inches="tight")
plt.show()

The vast majority of cells are zero. Each document only touches a handful of the available dimensions. In a real corpus with a 50,000-word vocabulary, a typical document might use 200 words — meaning 99.6% of each row is zeros.

This extreme sparsity is why methods like TF-IDF, word embeddings, and topic models exist — they compress the huge sparse space into a dense, meaningful, lower-dimensional representation.

## Patient Records

Electronic Health Records (EHRs) are high-dimensional by nature. A single admission might record dozens of lab values, vital signs, medication administrations, diagnostic codes, and clinical notes. But most of these features are heavily correlated — when one organ system fails, many lab values move together.

In [ ]:
# Generate synthetic patient data with 5 signal dims + 45 noise dims
X_patients, feature_names, signal_mask = make_patient_data(
    n=200, d_signal=5, d_noise=45, seed=42
)

print(f"Patient data: {X_patients.shape[0]} patients × {X_patients.shape[1]} features")
print(f"Signal features: {[n for n, s in zip(feature_names, signal_mask) if s]}")
print(f"Noise features: {sum(~signal_mask)} random columns\n")

# Correlation matrix — where's the redundancy?
corr = np.corrcoef(X_patients.T)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Correlation Matrix: 50 features, but most are noise")
ax.axhline(4.5, color="black", linewidth=1, linestyle="--")
ax.axvline(4.5, color="black", linewidth=1, linestyle="--")
ax.text(2, -2, "Signal", ha="center", fontsize=10, fontweight="bold", color=COLOURS["signal"])
ax.text(27, -2, "Noise", ha="center", fontsize=10, fontweight="bold", color=COLOURS["noise"])
plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.savefig("visuals/01c_patient_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# PCA: how many components actually matter?
pca_patients = PCA(random_state=42).fit(X_patients)
cumvar = np.cumsum(pca_patients.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, "o-",
        color=COLOURS["signal"], markersize=4, linewidth=2)
ax.axhline(y=0.95, color=COLOURS["accent"], linestyle="--", alpha=0.7, label="95% threshold")
ax.set_xlabel("Number of Components")
ax.set_ylabel("Cumulative Explained Variance")
ax.set_title("50 features, but how many matter?")
ax.legend()
ax.set_ylim(0, 1.05)

n_95 = np.searchsorted(cumvar, 0.95) + 1
ax.axvline(n_95, color=COLOURS["accent"], linewidth=0.8, alpha=0.5)
ax.annotate(f"{n_95} components for 95%", (n_95, 0.95),
            textcoords="offset points", xytext=(15, -15),
            fontsize=10, color=COLOURS["accent"],
            arrowprops=dict(arrowstyle="->", color=COLOURS["accent"]))

plt.tight_layout()
plt.show()

print(f"{n_95} out of 50 components capture 95% of the variance.")
print("The other dimensions are noise — they add columns but not information.")

This is exactly what happens in real clinical data. A doctor looking at 50 lab values doesn't process them as 50 independent numbers. They pattern-match: "this looks like sepsis" or "this looks like heart failure." That's dimensionality reduction — compressing 50 features into a handful of clinical concepts.

The methods in Modules 03–06 do the same thing, but rigorously and at scale.

## Simulation Parameter Spaces

Operational simulations — wargames, logistics models, casualty evacuation planners — have parameter spaces that are modest by genomics standards but still high enough for the curse to bite.

Consider a casualty evacuation scenario with 15 input parameters:

In [ ]:
# A simple simulation scenario with 15 parameters
sim_params = {
    "n_casualties": "Number of casualties (5–50)",
    "severity_mix": "Fraction T1 vs T2 vs T3 (0–1)",
    "n_helicopters": "Available helicopters (1–8)",
    "helo_speed_kts": "Helicopter speed in knots (80–150)",
    "dist_to_role2_km": "Distance to Role 2 facility (10–200 km)",
    "dist_to_role3_km": "Distance to Role 3 facility (50–500 km)",
    "weather_ceiling_ft": "Cloud ceiling (500–5000 ft)",
    "visibility_km": "Visibility (1–20 km)",
    "threat_level": "Enemy threat level (0–1)",
    "n_medics": "Number of medics at POI (1–10)",
    "golden_hour_min": "Target time to surgery (30–90 min)",
    "terrain_factor": "Terrain difficulty multiplier (1–3)",
    "night_ops": "Night operations (0 or 1)",
    "comms_reliability": "Communications reliability (0.5–1.0)",
    "surge_capacity": "Role 2 surge capacity (10–100 beds)",
}

print(f"{len(sim_params)} parameters = {len(sim_params)} dimensions\n")
for name, desc in sim_params.items():
    print(f"  {name:.<25s} {desc}")

# The grid search problem
levels = 5
grid_size = levels ** len(sim_params)
print(f"\nGrid search at {levels} levels per dimension: {levels}^{len(sim_params)} = {grid_size:,.0f} scenarios")
print(f"That's {grid_size:.1e} — roughly 30 billion runs.")
print(f"At 1 second per run, that's {grid_size / (3600 * 24 * 365):.0f} years of compute time.")

In [ ]:
# Latin Hypercube vs Grid: efficient coverage of the space
from scipy.stats import qmc

n_samples = 200
d = 2  # visualise in 2D for clarity

# Grid sampling
grid_1d = np.linspace(0, 1, int(np.sqrt(n_samples)))
grid_x, grid_y = np.meshgrid(grid_1d, grid_1d)

# Latin Hypercube sampling
sampler = qmc.LatinHypercube(d=2, seed=42)
lhs_points = sampler.random(n=n_samples)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(grid_x.ravel(), grid_y.ravel(), s=15, alpha=0.6, color=COLOURS["noise"])
ax1.set_title(f"Grid Sampling ({len(grid_x.ravel())} points)")
ax1.set_xlabel("Parameter 1")
ax1.set_ylabel("Parameter 2")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

ax2.scatter(lhs_points[:, 0], lhs_points[:, 1], s=15, alpha=0.6, color=COLOURS["signal"])
ax2.set_title(f"Latin Hypercube Sampling ({n_samples} points)")
ax2.set_xlabel("Parameter 1")
ax2.set_ylabel("Parameter 2")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

fig.suptitle("Smarter sampling covers the space with far fewer points", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/01c_sampling_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("In 2D, grid and LHS look similar.")
print(f"In 15D, a grid needs {5**15:,.0f} points. LHS can cover the space with ~500.")

This is where the curse meets operational reality. You can't run 30 billion scenarios, so you need to be smart about which regions of the parameter space to explore. That requires understanding which parameters actually matter — which is, once again, a dimensionality reduction problem.

## The Common Thread

Look at what we've seen across four very different domains:

| Domain | Ambient Dimension | What's Really Going On |
|--------|------------------|----------------------|
| Digit images | 64 pixels | ~10 digit classes with smooth variations |
| Text (bag-of-words) | 50,000+ words | A few hundred topics |
| Patient records | 500+ features | A handful of disease processes |
| Simulation parameters | 15–100 | A few critical factors dominate outcomes |

Every domain has the same pattern:

1. The data **looks** like it needs hundreds or thousands of dimensions
2. It **actually** lives on a much smaller structure (a manifold, a few clusters, a handful of factors)
3. The gap between ambient and intrinsic dimension is where dimensionality reduction lives

Finding that structure is the entire game. That's what Module 02 starts to teach you — and the rest of this repo builds on.

<details>
<summary><b>The Maths Behind This</b></summary>

The pattern we're seeing is formalised as the **manifold hypothesis**: real-world high-dimensional data tends to concentrate near a lower-dimensional manifold embedded in the ambient space.

If data in $\mathbb{R}^D$ actually lies on or near a $d$-dimensional manifold where $d \ll D$, then:
- The intrinsic dimension is $d$, not $D$
- The curse of dimensionality applies to $d$, not $D$
- Algorithms that can discover this manifold can work in $d$ dimensions instead of $D$

This is why PCA captures 95% of variance with a few components, why word embeddings compress 50,000D to 300D, and why clinical experts can summarise 500 features as "this patient is septic."

</details>

## Where This Matters

**Every field with data.** The examples in this notebook aren't special cases — they're the norm. Genomics, radiology, NLP, financial modelling, sensor networks, drug discovery — all of them face the same ambient-vs-intrinsic dimension gap.

The practical consequence: before running any algorithm on high-dimensional data, ask yourself: *what's the intrinsic dimension here?* If you don't know, you're probably fighting the curse instead of exploiting the structure.

## Feynman Check

Can you explain these without reaching for jargon?

1. **Give three real-world examples of data with more than 100 dimensions.**

2. **Why are most entries in a bag-of-words matrix zero?**

3. **When a doctor looks at 50 lab values and says "this patient is septic," what kind of dimensionality reduction are they doing?**

Module 01 is complete. You now understand what dimensions are, why they're dangerous, and where they show up in the real world. In **Module 02: Redundancy and Structure**, we'll start learning how to exploit the gap between ambient and intrinsic dimension — the first step toward fighting back.